# 🧠 Neural Implicit SDF — 3D Medical Reconstruction

**Pipeline:**
```
NIfTI mask → Isotropic resample → Compute SDF → Train SIREN MLP → Extract mesh
```

**Tại sao tốt hơn Marching Cubes thông thường:**
- Không bị bậc thang voxel (staircase artifact)
- Sub-voxel accuracy — chi tiết nhỏ hơn 1mm
- Extract mesh ở resolution tuỳ ý (2x, 4x gốc)
- Topology mượt hơn về mặt toán học

**Thời gian train ước tính trên Kaggle GPU (T4):**
- Kidney (~359k voxels): ~8–12 phút
- Tumor/Cyst (nhỏ hơn): ~3–5 phút mỗi cái


In [ ]:
# ============================================================
# CELL 1 — Cài đặt
# torch đã có sẵn trên Kaggle, chỉ cần thêm:
# ============================================================
!pip install -q nibabel pyvista trimesh plotly
print('✅ Done')

In [ ]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.ndimage import distance_transform_edt, zoom
import pyvista as pv
import trimesh
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ============================================================
# CELL 3 — Config
# ============================================================
NII_PATH   = "/kaggle/input/datasets/khoayakult/data-khoa/2_case_00066_pred.nii"
OUTPUT_DIR = "/kaggle/working"
case_name  = os.path.basename(NII_PATH).replace('.nii.gz','').replace('.nii','')
HTML_OUT   = os.path.join(OUTPUT_DIR, f"3d_{case_name}_neural_sdf.html")

assert os.path.exists(NII_PATH), f"Không tìm thấy: {NII_PATH}"
print(f'✅ File: {NII_PATH}')

In [ ]:
# ============================================================
# CELL 4 — Load + Isotropic Resample (scipy zoom, không cần MONAI)
# ============================================================
from scipy.ndimage import zoom

print('Loading NIfTI...')
nii          = nib.load(NII_PATH)
seg_raw      = nii.get_fdata(dtype=np.float32)
spacing_orig = np.array(nii.header.get_zooms())

print(f'  Shape gốc   : {seg_raw.shape}')
print(f'  Spacing     : {spacing_orig} mm')

# Resample về 1×1×1 mm
zoom_factors = spacing_orig / 1.0
print(f'  Zoom factors: {zoom_factors}')

seg_iso = zoom(seg_raw.astype(np.uint8), zoom=zoom_factors,
               order=0, prefilter=False).astype(np.uint8)
print(f'  Shape iso   : {seg_iso.shape}')
print(f'  Labels      : {np.unique(seg_iso).tolist()}')

# Auto-crop
fg = np.where(seg_iso > 0)
M  = 15
x0,x1 = max(0,fg[0].min()-M), min(seg_iso.shape[0],fg[0].max()+M)
y0,y1 = max(0,fg[1].min()-M), min(seg_iso.shape[1],fg[1].max()+M)
z0,z1 = max(0,fg[2].min()-M), min(seg_iso.shape[2],fg[2].max()+M)
cropped = seg_iso[x0:x1, y0:y1, z0:z1]
print(f'  Shape crop  : {cropped.shape}')

In [ ]:
# ============================================================
# CELL 5 — Hàm tính SDF từ binary mask
#
# scipy.ndimage.distance_transform_edt tính Euclidean distance
# SDF convention:
#   +d bên trong mask (dương = inside organ)
#   -d bên ngoài mask (âm  = outside organ)
# Surface chính xác = tập hợp điểm SDF = 0
# ============================================================
def compute_sdf(binary_mask: np.ndarray) -> np.ndarray:
    """
    Tính Signed Distance Function từ binary mask.
    Trả về array cùng shape, giá trị tính bằng mm (sau isotropic resample).
    """
    mask_bool = binary_mask.astype(bool)

    # Khoảng cách từ mỗi điểm đến surface (luôn dương)
    dist_in  = distance_transform_edt(mask_bool)       # trong mask → dương
    dist_out = distance_transform_edt(~mask_bool)      # ngoài mask → dương

    # Ghép thành SDF có dấu
    sdf = dist_in - dist_out  # dương bên trong, âm bên ngoài
    return sdf.astype(np.float32)


def normalize_coords(shape):
    """
    Tạo lưới toạ độ normalized [-1, 1] cho volume shape (nx, ny, nz).
    Trả về array (nx*ny*nz, 3).
    """
    nx, ny, nz = shape
    xs = np.linspace(-1, 1, nx, dtype=np.float32)
    ys = np.linspace(-1, 1, ny, dtype=np.float32)
    zs = np.linspace(-1, 1, nz, dtype=np.float32)
    gx, gy, gz = np.meshgrid(xs, ys, zs, indexing='ij')
    coords = np.stack([gx, gy, gz], axis=-1).reshape(-1, 3)
    return coords


print('✅ SDF utilities ready')

In [ ]:
# ============================================================
# CELL 6 — SIREN Network
#
# SIREN = Sinusoidal Representation Network (Sitzmann et al. 2020)
# Dùng sin() thay ReLU → học được high-frequency details tốt hơn
# Đặc biệt phù hợp cho SDF fitting của shape phức tạp
#
# Architecture:
#   Input (3) → [Linear + Sin] × 4 → Linear (1)
#   Hidden dim = 256 (đủ cho organ phức tạp)
# ============================================================
class SIRENLayer(nn.Module):
    def __init__(self, in_f, out_f, is_first=False, omega_0=30.0):
        super().__init__()
        self.omega_0  = omega_0
        self.is_first = is_first
        self.linear   = nn.Linear(in_f, out_f)
        self._init_weights(in_f)

    def _init_weights(self, in_f):
        # SIREN-specific initialization (quan trọng — random thường không hội tụ)
        with torch.no_grad():
            if self.is_first:
                bound = 1.0 / in_f
            else:
                bound = np.sqrt(6.0 / in_f) / self.omega_0
            self.linear.weight.uniform_(-bound, bound)
            self.linear.bias.zero_()

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class SDFNet(nn.Module):
    """
    MLP học SDF từ toạ độ 3D.
    Input:  (B, 3) toạ độ normalized [-1, 1]
    Output: (B, 1) giá trị SDF normalized
    """
    def __init__(self, hidden_dim=256, n_layers=4, omega_0=30.0):
        super().__init__()
        layers = [SIRENLayer(3, hidden_dim, is_first=True, omega_0=omega_0)]
        for _ in range(n_layers - 1):
            layers.append(SIRENLayer(hidden_dim, hidden_dim, omega_0=omega_0))
        self.net    = nn.Sequential(*layers)
        self.output = nn.Linear(hidden_dim, 1)

        # Init output layer nhỏ để bắt đầu gần 0
        nn.init.uniform_(self.output.weight, -np.sqrt(6/hidden_dim)/omega_0,
                                              np.sqrt(6/hidden_dim)/omega_0)
        nn.init.zeros_(self.output.bias)

    def forward(self, x):
        return self.output(self.net(x))   # (B, 1)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Test nhanh
test_net = SDFNet(hidden_dim=256, n_layers=4).to(DEVICE)
print(f'✅ SDFNet params: {count_params(test_net):,}')
x_test = torch.randn(8, 3).to(DEVICE)
print(f'   Forward test : input {x_test.shape} → output {test_net(x_test).shape}')

In [ ]:
# ============================================================
# CELL 7 — Hàm sample training data từ SDF
#
# Strategy lấy mẫu (quan trọng cho chất lượng):
#   50% near-surface  → điểm sát biên, MLP học surface chính xác
#   30% random volume → MLP học SDF toàn bộ không gian
#   20% on-surface    → điểm ngay trên surface (SDF ≈ 0)
# ============================================================
def sample_training_points(sdf_volume: np.ndarray,
                           n_samples: int = 200_000,
                           near_surface_band: float = 5.0) -> tuple:
    """
    Sample toạ độ + SDF values để train MLP.

    Args:
        sdf_volume       : (nx, ny, nz) SDF values (mm)
        n_samples        : tổng số điểm lấy mẫu
        near_surface_band: lấy điểm trong vòng ±band mm từ surface

    Returns:
        coords_norm : (N, 3) normalized [-1,1]
        sdf_norm    : (N,)   SDF normalized by max_dist
    """
    nx, ny, nz = sdf_volume.shape
    max_dist   = float(np.abs(sdf_volume).max()) + 1e-6

    # Tạo lưới index
    xi = np.arange(nx, dtype=np.float32)
    yi = np.arange(ny, dtype=np.float32)
    zi = np.arange(nz, dtype=np.float32)

    # Near-surface mask
    near_mask  = np.abs(sdf_volume) < near_surface_band
    near_idx   = np.argwhere(near_mask)

    n_near   = int(n_samples * 0.50)
    n_rand   = int(n_samples * 0.30)
    n_surf   = n_samples - n_near - n_rand

    all_coords = []
    all_sdf    = []

    # ── 1) Near-surface samples ──────────────────────────────
    if len(near_idx) > 0:
        idx = near_idx[np.random.choice(len(near_idx),
                                        min(n_near, len(near_idx)),
                                        replace=True)]
        # Thêm jitter nhỏ để không bị aliased tại grid points
        jitter = np.random.randn(*idx.shape).astype(np.float32) * 0.5
        idx_j  = np.clip(idx.astype(np.float32) + jitter,
                         [0,0,0], [nx-1, ny-1, nz-1]).astype(int)
        all_coords.append(idx_j)
        all_sdf.append(sdf_volume[idx_j[:,0], idx_j[:,1], idx_j[:,2]])

    # ── 2) Random volume samples ─────────────────────────────
    rand_idx = np.stack([
        np.random.randint(0, nx, n_rand),
        np.random.randint(0, ny, n_rand),
        np.random.randint(0, nz, n_rand),
    ], axis=1)
    all_coords.append(rand_idx)
    all_sdf.append(sdf_volume[rand_idx[:,0], rand_idx[:,1], rand_idx[:,2]])

    # ── 3) On-surface: điểm gần zero crossing nhất ───────────
    surf_mask = np.abs(sdf_volume) < 1.5   # rất sát surface
    surf_idx  = np.argwhere(surf_mask)
    if len(surf_idx) > 0:
        idx = surf_idx[np.random.choice(len(surf_idx),
                                        min(n_surf, len(surf_idx)),
                                        replace=True)]
        all_coords.append(idx)
        all_sdf.append(sdf_volume[idx[:,0], idx[:,1], idx[:,2]])

    # ── Ghép + Normalize ──────────────────────────────────────
    coords_idx = np.concatenate(all_coords, axis=0).astype(np.float32)
    sdf_vals   = np.concatenate(all_sdf,    axis=0).astype(np.float32)

    # Normalize coords về [-1, 1]
    coords_norm = np.stack([
        coords_idx[:,0] / (nx - 1) * 2 - 1,
        coords_idx[:,1] / (ny - 1) * 2 - 1,
        coords_idx[:,2] / (nz - 1) * 2 - 1,
    ], axis=1)

    # Normalize SDF về [-1, 1]
    sdf_norm = sdf_vals / max_dist

    return coords_norm, sdf_norm, max_dist


print('✅ Sampling function ready')

In [ ]:
# ============================================================
# CELL 8 — Training function
#
# Loss = SDF_loss + lambda * Eikonal_loss
#
# SDF_loss   : L1 giữa predicted SDF và ground-truth SDF
#              L1 tốt hơn L2 vì ít bị outlier ảnh hưởng
#
# Eikonal    : ||∇SDF||₂ = 1 ở mọi điểm
#              Đây là tính chất toán học của SDF đúng nghĩa:
#              gradient luôn có độ dài = 1
#              Nếu không có term này MLP học được giá trị đúng
#              nhưng gradient sai → surface xấu khi extract
# ============================================================
def eikonal_loss(grad):
    """||∇f|| nên = 1 tại mọi điểm (tính chất SDF đúng)."""
    return ((grad.norm(dim=-1) - 1.0) ** 2).mean()


def train_sdf_net(model, coords_norm, sdf_norm,
                  n_epochs=500, batch_size=16384,
                  lr=1e-4, lambda_eik=0.1,
                  print_every=50):
    """
    Train SDFNet trên (coords, sdf_values).
    Trả về model đã train và loss history.
    """
    coords_t = torch.from_numpy(coords_norm).to(DEVICE)  # (N, 3)
    sdf_t    = torch.from_numpy(sdf_norm).to(DEVICE)     # (N,)

    dataset  = TensorDataset(coords_t, sdf_t)
    loader   = DataLoader(dataset, batch_size=batch_size,
                          shuffle=True, drop_last=True)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    # Cosine annealing: lr giảm dần → fine-tune ở cuối
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=lr * 0.01
    )

    model.train()
    history = []

    for epoch in range(1, n_epochs + 1):
        epoch_loss = 0.0
        n_batches  = 0

        for batch_coords, batch_sdf in loader:
            batch_coords.requires_grad_(True)

            pred = model(batch_coords).squeeze(-1)  # (B,)

            # ── SDF loss (L1) ──────────────────────────────
            loss_sdf = torch.nn.functional.l1_loss(pred, batch_sdf)

            # ── Eikonal loss ───────────────────────────────
            grad = torch.autograd.grad(
                outputs=pred.sum(),
                inputs=batch_coords,
                create_graph=True,
            )[0]  # (B, 3)
            loss_eik = eikonal_loss(grad)

            loss = loss_sdf + lambda_eik * loss_eik

            optimizer.zero_grad()
            loss.backward()
            # Gradient clipping: tránh exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            n_batches  += 1

        scheduler.step()
        avg_loss = epoch_loss / max(n_batches, 1)
        history.append(avg_loss)

        if epoch % print_every == 0 or epoch == 1:
            lr_now = scheduler.get_last_lr()[0]
            print(f'  Epoch {epoch:4d}/{n_epochs} | loss={avg_loss:.5f} | lr={lr_now:.2e}')

    return model, history


print('✅ Training function ready')

In [ ]:
# ============================================================
# CELL 9 — Mesh extraction từ trained SDF model
#
# Query MLP trên lưới 3D → SDF volume → Marching Cubes tại level=0
# upscale_factor=2 → extract ở 2x resolution so với voxel gốc
#                    → sub-voxel accuracy
# ============================================================
@torch.no_grad()
def extract_mesh_from_sdf_net(model, original_shape,
                               upscale_factor=2,
                               batch_size=50000):
    """
    Query trained SDFNet trên lưới 3D để extract mesh.

    Args:
        model          : trained SDFNet
        original_shape : (nx, ny, nz) shape của volume gốc
        upscale_factor : resolution multiplier (2 = 2x mịn hơn)
        batch_size     : số điểm query mỗi lần (VRAM dependent)

    Returns:
        PyVista PolyData mesh
    """
    model.eval()
    nx, ny, nz = original_shape

    # Lưới ở độ phân giải cao hơn
    nx2 = int(nx * upscale_factor)
    ny2 = int(ny * upscale_factor)
    nz2 = int(nz * upscale_factor)

    print(f'  Query grid: {nx2}×{ny2}×{nz2} = {nx2*ny2*nz2:,} points')

    xs = np.linspace(-1, 1, nx2, dtype=np.float32)
    ys = np.linspace(-1, 1, ny2, dtype=np.float32)
    zs = np.linspace(-1, 1, nz2, dtype=np.float32)
    gx, gy, gz = np.meshgrid(xs, ys, zs, indexing='ij')
    all_coords = np.stack([gx, gy, gz], axis=-1).reshape(-1, 3)  # (N, 3)

    # Query MLP theo batch để không OOM
    sdf_pred = np.zeros(len(all_coords), dtype=np.float32)
    n_total  = len(all_coords)

    for start in range(0, n_total, batch_size):
        end   = min(start + batch_size, n_total)
        batch = torch.from_numpy(all_coords[start:end]).to(DEVICE)
        out   = model(batch).squeeze(-1).cpu().numpy()
        sdf_pred[start:end] = out

        if start % (batch_size * 20) == 0:
            pct = start / n_total * 100
            print(f'  Querying... {pct:.0f}%', end='\r')

    print(f'  Querying... 100% ✓')

    # Reshape về 3D
    sdf_vol = sdf_pred.reshape(nx2, ny2, nz2)

    # Marching Cubes tại SDF = 0 (surface chính xác)
    grid = pv.ImageData()
    grid.dimensions = (nx2, ny2, nz2)
    # Mapping về mm space: [-1,1] → [0, original_size_mm]
    grid.spacing = (
        nx / (nx2 - 1),
        ny / (ny2 - 1),
        nz / (nz2 - 1),
    )
    grid.point_data['sdf'] = sdf_vol.flatten(order='F')
    mesh = grid.contour([0.0], scalars='sdf')   # level=0 = surface

    if mesh.n_points == 0:
        print('  ⚠️  Empty mesh — kiểm tra lại SDF hoặc tăng n_epochs')
        return None

    print(f'  Mesh: {mesh.n_points:,} verts | {mesh.n_faces:,} faces')
    return mesh


print('✅ Mesh extraction function ready')

In [ ]:
# ============================================================
# CELL 10 — Hàm tiện ích Plotly (giữ nguyên từ pipeline cũ)
# ============================================================
def pyvista_to_plotly(pv_mesh, color, opacity, name):
    verts      = np.array(pv_mesh.points)
    tri_mesh   = pv_mesh.triangulate()
    faces_flat = tri_mesh.faces.reshape(-1, 4)[:, 1:]
    return go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces_flat[:,0], j=faces_flat[:,1], k=faces_flat[:,2],
        color=color, opacity=opacity, name=name,
        showscale=False, flatshading=False,
        lighting=dict(ambient=0.45, diffuse=0.85,
                      specular=0.35, roughness=0.45, fresnel=0.25),
        lightposition=dict(x=100, y=200, z=300),
    )


print('✅ Plotly helper ready')

In [ ]:
# ============================================================
# CELL 11 — MAIN LOOP: Train SDF + Extract mesh cho từng label
#
# Config per-label:
#   n_epochs    : nhiều hơn = mượt hơn, lâu hơn
#   n_samples   : nhiều hơn = chính xác hơn, tốn RAM hơn
#   upscale     : 2 = 2x resolution, đủ cho hầu hết cases
#   hidden_dim  : 256 đủ cho organ; 512 cho shapes rất phức tạp
# ============================================================
PARTS = [
    {'label': 1, 'name': 'Than (Kidney)',  'color': '#E8735A', 'opacity': 0.35,
     'n_epochs': 600, 'n_samples': 300_000, 'upscale': 2,
     'hidden_dim': 256, 'n_layers': 5, 'min_faces': 200},

    {'label': 2, 'name': 'Khoi U (Tumor)', 'color': '#FFD700', 'opacity': 0.95,
     'n_epochs': 400, 'n_samples': 150_000, 'upscale': 2,
     'hidden_dim': 256, 'n_layers': 4, 'min_faces': 30},

    {'label': 3, 'name': 'Nang (Cyst)',    'color': '#00CED1', 'opacity': 0.90,
     'n_epochs': 400, 'n_samples': 150_000, 'upscale': 2,
     'hidden_dim': 256, 'n_layers': 4, 'min_faces': 30},

    {'label': 4, 'name': 'Label 4',        'color': '#AA00FF', 'opacity': 0.90,
     'n_epochs': 300, 'n_samples': 100_000, 'upscale': 2,
     'hidden_dim': 128, 'n_layers': 4, 'min_faces': 20},

    {'label': 5, 'name': 'Label 5',        'color': '#00FF88', 'opacity': 0.90,
     'n_epochs': 300, 'n_samples': 100_000, 'upscale': 2,
     'hidden_dim': 128, 'n_layers': 4, 'min_faces': 20},
]

fig        = go.Figure()
summary    = []
meshes_all = []
colors_all = []

for part in PARTS:
    lab  = part['label']
    name = part['name']
    print(f"\n{'='*58}")
    print(f"  {name}  (label={lab})")
    print(f"{'='*58}")

    mask  = (cropped == lab).astype(np.uint8)
    n_vox = int(mask.sum())

    if n_vox == 0:
        print(f'  ⚠️  Không có voxel. Bỏ qua.')
        continue

    print(f'  Voxels: {n_vox:,}')

    # ── BƯỚC 1: Tính SDF ─────────────────────────────────────
    print('  [1/4] Computing SDF...')
    sdf_vol = compute_sdf(mask)
    max_sdf = float(np.abs(sdf_vol).max())
    print(f'        SDF range: [{sdf_vol.min():.1f}, {sdf_vol.max():.1f}] mm')

    # ── BƯỚC 2: Sample training points ───────────────────────
    print('  [2/4] Sampling training points...')
    coords_norm, sdf_norm, max_dist = sample_training_points(
        sdf_vol,
        n_samples=part['n_samples'],
        near_surface_band=5.0,
    )
    print(f'        {len(coords_norm):,} training points')

    # ── BƯỚC 3: Train SIREN ───────────────────────────────────
    print(f'  [3/4] Training SIREN ({part["n_epochs"]} epochs)...')
    model = SDFNet(
        hidden_dim=part['hidden_dim'],
        n_layers=part['n_layers'],
    ).to(DEVICE)

    model, history = train_sdf_net(
        model, coords_norm, sdf_norm,
        n_epochs=part['n_epochs'],
        batch_size=16384,
        lr=1e-4,
        lambda_eik=0.1,
        print_every=100,
    )
    print(f'        Final loss: {history[-1]:.5f}')

    # Lưu model (có thể load lại sau)
    model_path = os.path.join(OUTPUT_DIR, f'sdf_model_label{lab}.pt')
    torch.save(model.state_dict(), model_path)
    print(f'        Model saved: {model_path}')

    # ── BƯỚC 4: Extract mesh ──────────────────────────────────
    print(f'  [4/4] Extracting mesh (upscale={part["upscale"]}x)...')
    mesh = extract_mesh_from_sdf_net(
        model,
        original_shape=cropped.shape,
        upscale_factor=part['upscale'],
        batch_size=50000,
    )

    if mesh is None:
        print('  ❌ Mesh extraction failed. Bỏ qua.')
        continue

    # Lọc component nhỏ bằng trimesh
    verts_np = np.array(mesh.triangulate().points)
    faces_np = mesh.triangulate().faces.reshape(-1, 4)[:, 1:]
    tm       = trimesh.Trimesh(vertices=verts_np, faces=faces_np, process=False)
    comps    = tm.split(only_watertight=False)
    large    = [c for c in comps if len(c.faces) >= part['min_faces']]
    if not large:
        large = [max(comps, key=lambda c: len(c.faces))]
    merged = trimesh.util.concatenate(large) if len(large) > 1 else large[0]
    mesh   = pv.wrap(merged).triangulate()

    print(f'  Final: {mesh.n_points:,} verts | {mesh.n_faces:,} faces | '
          f'{len(large)}/{len(comps)} components')

    # Stats
    vol_mm3 = round(float(mesh.volume), 2) if mesh.is_manifold else None
    summary.append({
        'name': name,
        'vertices': mesh.n_points,
        'faces': mesh.n_faces,
        'surface_area_mm2': round(float(mesh.area), 2),
        'volume_mm3': vol_mm3,
        'final_loss': round(history[-1], 5),
    })

    fig.add_trace(pyvista_to_plotly(mesh, part['color'], part['opacity'], name))
    meshes_all.append(mesh)
    colors_all.append(part['color'])
    print(f'  ✅ Done {name}')

print(f"\n{'='*58}")
print('Tất cả labels xong!')

In [ ]:
# ============================================================
# CELL 12 — Summary + Export HTML
# ============================================================
print('\n📊 SUMMARY')
print(f"{'─'*75}")
print(f"{'Tên':<20} {'Verts':>8} {'Faces':>8} {'SA mm²':>10} {'Vol mm³':>10} {'Loss':>8}")
print(f"{'─'*75}")
for s in summary:
    v = f"{s['volume_mm3']:,.0f}" if s['volume_mm3'] else 'N/A'
    print(f"{s['name']:<20} {s['vertices']:>8,} {s['faces']:>8,} "
          f"{s['surface_area_mm2']:>10,.1f} {v:>10} {s['final_loss']:>8.5f}")
print(f"{'─'*75}")

fig.update_layout(
    title=dict(
        text=f'Neural SDF 3D Reconstruction | {case_name}',
        x=0.5, font=dict(size=16)
    ),
    scene=dict(
        xaxis=dict(title='X (mm)', showbackground=True, backgroundcolor='#2a2a3e'),
        yaxis=dict(title='Y (mm)', showbackground=True, backgroundcolor='#252535'),
        zaxis=dict(title='Z (mm)', showbackground=True, backgroundcolor='#2a2a3e'),
        aspectmode='data',
        bgcolor='#12121f',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)),
    ),
    paper_bgcolor='#12121f',
    font=dict(color='white'),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(0,0,0,0.6)',
                font=dict(color='white', size=12)),
    width=1100, height=900,
    margin=dict(r=10, l=10, b=10, t=55),
)

fig.write_html(HTML_OUT, include_plotlyjs='cdn', full_html=True)
print(f'\n✅ HTML: {HTML_OUT} ({os.path.getsize(HTML_OUT)/1e6:.1f} MB)')

In [ ]:
# ============================================================
# CELL 13 — Export GLB
# ============================================================
if meshes_all:
    def hex_to_rgba(h):
        h = h.lstrip('#')
        return [int(h[i:i+2],16)/255. for i in (0,2,4)] + [1.0]

    scene = trimesh.scene.Scene()
    for pv_mesh, chex in zip(meshes_all, colors_all):
        verts = np.array(pv_mesh.points)
        faces = pv_mesh.faces.reshape(-1,4)[:,1:]
        tm    = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
        tm.visual = trimesh.visual.ColorVisuals(
            mesh=tm,
            vertex_colors=np.tile(hex_to_rgba(chex), (len(verts),1)),
        )
        scene.add_geometry(tm)

    glb_path = HTML_OUT.replace('.html', '.glb')
    with open(glb_path, 'wb') as f:
        f.write(scene.export(file_type='glb'))
    print(f'✅ GLB: {glb_path} ({os.path.getsize(glb_path)/1e6:.1f} MB)')

print('\n📁 All outputs:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    print(f'   {f:<55} {os.path.getsize(fp)/1e6:.1f} MB')